# 📦 RAG Module — Data Ingestion Pipeline
### AI/ML Bootcamp — Day 2

This notebook covers the **first half of a RAG system** — everything that happens *before* a user asks a question.

```
┌─────────────────────────────────────────────────────────────────────┐
│                    DATA INGESTION PIPELINE                          │
│                                                                     │
│  Raw Files  →  Document Loaders  →  Chunking  →  Embeddings  →  Vector DB │
│  (.txt, .pdf,    (format-aware      (split into   (encode to    (store &   │
│   .docx, .csv,    parsers)           small pieces)  vectors)      index)    │
│   .json)                                                                    │
└─────────────────────────────────────────────────────────────────────┘
```

**Sections:**
1. Installation & Setup
2. Document Loaders (format-aware parsing)
3. Chunking Strategies (5 strategies with pros/cons)
4. Embeddings (all-MiniLM-L6-v2 + model comparison)
5. Vector Databases (ChromaDB vs FAISS)
6. Full Pipeline — putting it all together

**Sample data:** 5 files about a fictional company NovaTech AI
- `novatech_company.txt` — company overview
- `novatech_faq.json` — product FAQs
- `novatech_employees.csv` — employee directory
- `novatech_handbook.docx` — product handbook
- `novatech_architecture.pdf` — technical architecture

---

## Section 1 — Installation & Setup

In [ ]:
# Install everything needed for the full pipeline
!pip install \
    langchain langchain-community langchain-text-splitters \
    sentence-transformers \
    chromadb \
    faiss-cpu \
    pypdf \
    python-docx \
    unstructured \
    rich \
    matplotlib numpy pandas -q

print("✅ All packages installed")

In [ ]:
import os, json, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from rich import print as rprint
from rich.table import Table
from rich.panel import Panel

# ── Set your data directory ───────────────────────────────────
DATA_DIR = "./sample_data"   # adjust if needed
# ─────────────────────────────────────────────────────────────

# Verify files exist
expected_files = [
    "novatech_company.txt",
    "novatech_faq.json",
    "novatech_employees.csv",
    "novatech_handbook.docx",
    "novatech_architecture.pdf",
]

print("Checking sample data files:")
for f in expected_files:
    path = Path(DATA_DIR) / f
    status = "✅" if path.exists() else "❌ MISSING"
    size   = f"{path.stat().st_size / 1024:.1f} KB" if path.exists() else ""
    print(f"  {status}  {f:40s}  {size}")

---
## Section 2 — Document Loaders

Document loaders are **format-aware parsers** that extract raw text from files.
Different file formats need different loaders — a PDF parser won't work on a CSV.

We use **LangChain's document loaders** which standardize all formats into a common
`Document` object: `{ page_content: str, metadata: dict }`.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.1  TXT Loader — plain text files
# ─────────────────────────────────────────────────────────────
from langchain_community.document_loaders import TextLoader

txt_loader = TextLoader(
    file_path=f"{DATA_DIR}/novatech_company.txt",
    encoding="utf-8"
)
txt_docs = txt_loader.load()

print("=== TXT Loader ===")
print(f"Documents loaded : {len(txt_docs)}")
print(f"Characters       : {len(txt_docs[0].page_content):,}")
print(f"Metadata         : {txt_docs[0].metadata}")
print(f"\nFirst 300 chars  :\n{txt_docs[0].page_content[:300]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.2  PDF Loader — PyPDFLoader splits by page automatically
# ─────────────────────────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader(f"{DATA_DIR}/novatech_architecture.pdf")
pdf_docs   = pdf_loader.load()

print("=== PDF Loader ===")
print(f"Pages loaded  : {len(pdf_docs)}")
for i, page in enumerate(pdf_docs):
    print(f"\n--- Page {i+1} ---")
    print(f"Characters : {len(page.page_content):,}")
    print(f"Metadata   : {page.metadata}")
    print(f"Preview    : {page.page_content[:200]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.3  DOCX Loader — extracts text from Word documents
# ─────────────────────────────────────────────────────────────
from langchain_community.document_loaders import Docx2txtLoader

docx_loader = Docx2txtLoader(f"{DATA_DIR}/novatech_handbook.docx")
docx_docs   = docx_loader.load()

print("=== DOCX Loader ===")
print(f"Documents loaded : {len(docx_docs)}")
print(f"Characters       : {len(docx_docs[0].page_content):,}")
print(f"Metadata         : {docx_docs[0].metadata}")
print(f"\nPreview:\n{docx_docs[0].page_content[:400]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.4  CSV Loader — each row becomes a Document
# ─────────────────────────────────────────────────────────────
from langchain_community.document_loaders.csv_loader import CSVLoader

csv_loader = CSVLoader(
    file_path=f"{DATA_DIR}/novatech_employees.csv",
    source_column="name"   # used as the source in metadata
)
csv_docs = csv_loader.load()

print("=== CSV Loader ===")
print(f"Rows loaded (each row = 1 Document): {len(csv_docs)}")
print(f"\nFirst 3 documents:")
for doc in csv_docs[:3]:
    print(f"\n  Content  : {doc.page_content}")
    print(f"  Metadata : {doc.metadata}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.5  JSON Loader — custom parsing for nested JSON
# LangChain's JSONLoader uses jq syntax to extract fields
# ─────────────────────────────────────────────────────────────

# Option A: Use LangChain JSONLoader with jq path
from langchain_community.document_loaders import JSONLoader

json_loader = JSONLoader(
    file_path=f"{DATA_DIR}/novatech_faq.json",
    jq_schema=".faqs[]",         # extract each FAQ object
    text_content=False            # don't try to stringify, use structured content
)

# Option B: Manual loader gives us more control over metadata
def load_json_as_documents(filepath: str):
    """Custom JSON loader that preserves category and id as metadata."""
    from langchain_core.documents import Document

    with open(filepath) as f:
        data = json.load(f)

    docs = []
    for item in data["faqs"]:
        # Combine Q+A into one piece of text for better retrieval
        content = f"Q: {item['question']}\nA: {item['answer']}"
        metadata = {
            "source"   : filepath,
            "faq_id"   : item["id"],
            "category" : item["category"],
        }
        docs.append(Document(page_content=content, metadata=metadata))
    return docs

json_docs = load_json_as_documents(f"{DATA_DIR}/novatech_faq.json")

print("=== JSON Loader (custom) ===")
print(f"FAQ documents loaded: {len(json_docs)}")
for doc in json_docs[:2]:
    print(f"\n  Metadata : {doc.metadata}")
    print(f"  Content  : {doc.page_content[:200]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.6  Combine all documents into one corpus
# ─────────────────────────────────────────────────────────────

all_docs = txt_docs + pdf_docs + docx_docs + csv_docs + json_docs

print(f"Total documents in corpus: {len(all_docs)}")
print(f"Total characters         : {sum(len(d.page_content) for d in all_docs):,}")
print()

# Summary table
table = Table(title="Document Corpus Summary", show_header=True)
table.add_column("Source",         style="cyan")
table.add_column("Format",         style="yellow")
table.add_column("Docs",           justify="right")
table.add_column("Total Chars",    justify="right")
table.add_column("Avg Chars/Doc",  justify="right")

groups = [
    ("novatech_company.txt",     "TXT",  txt_docs),
    ("novatech_architecture.pdf","PDF",  pdf_docs),
    ("novatech_handbook.docx",   "DOCX", docx_docs),
    ("novatech_employees.csv",   "CSV",  csv_docs),
    ("novatech_faq.json",        "JSON", json_docs),
]

for name, fmt, docs in groups:
    total_chars = sum(len(d.page_content) for d in docs)
    avg         = total_chars // len(docs) if docs else 0
    table.add_row(name, fmt, str(len(docs)), f"{total_chars:,}", f"{avg:,}")

rprint(table)

---
## Section 3 — Chunking Strategies

**Why do we chunk?**
- LLMs have a fixed context window — we can't feed them entire documents
- Smaller chunks produce more precise retrieval
- Embeddings of long texts lose specificity (one vector for 10,000 words is vague)

**The core trade-off:**
```
Smaller chunks  →  More precise retrieval,  but may lose context
Larger chunks   →  More context preserved,  but noisier retrieval
```

We'll implement and compare **5 chunking strategies**.

In [ ]:
# Use the TXT document as our test bed for chunking (long, varied content)
test_doc = txt_docs[0]
text     = test_doc.page_content

print(f"Test document: {test_doc.metadata['source']}")
print(f"Total characters: {len(text):,}")
print(f"Total words     : {len(text.split()):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.1  Fixed-Size Chunking (Character Splitter)
# ─────────────────────────────────────────────────────────────

"""
FIXED-SIZE CHUNKING
-------------------
How it works : Splits text every N characters. Overlap copies the last
               K characters into the next chunk to preserve context at boundaries.

PROS:
  + Simple and fast — no language understanding needed
  + Predictable chunk sizes → predictable embedding and retrieval costs
  + Works on any text regardless of structure

CONS:
  - Blindly cuts through sentences and paragraphs
  - A sentence split across two chunks may not retrieve well for either
  - Overlap adds redundant tokens, increasing storage costs

USE WHEN:
  * Your text has no meaningful structure (logs, raw scraped HTML)
  * You need a fast, simple baseline before trying better strategies
  * Token budget is the primary constraint
"""

from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    chunk_size=500,        # target chunk size in characters
    chunk_overlap=50,      # overlap between consecutive chunks
    separator="",          # no separator — purely character-based
    length_function=len,
)

fixed_chunks = fixed_splitter.create_documents(
    texts=[text],
    metadatas=[test_doc.metadata]
)

print("=== Fixed-Size Chunking ===")
print(f"Total chunks: {len(fixed_chunks)}")
sizes = [len(c.page_content) for c in fixed_chunks]
print(f"Avg size    : {sum(sizes)//len(sizes)} chars")
print(f"Min / Max   : {min(sizes)} / {max(sizes)} chars")
print(f"\nChunk 3 (shows mid-sentence cut):")
print(repr(fixed_chunks[2].page_content[:200]))

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.2  Recursive Character Splitter
# ─────────────────────────────────────────────────────────────

"""
RECURSIVE CHARACTER SPLITTING
------------------------------
How it works : Tries to split on a hierarchy of separators in order:
               ["\n\n", "\n", " ", ""]
               First tries double newlines (paragraphs), if chunk is still
               too large it falls back to single newlines, then spaces, then chars.

PROS:
  + Respects natural text boundaries (paragraphs > sentences > words)
  + Much cleaner splits than fixed-size — sentences stay intact
  + The DEFAULT recommended splitter for most RAG applications
  + Works well across different document types

CONS:
  - Chunk sizes are less uniform (some paragraphs are short, some long)
  - Still doesn't understand semantics — just structure

USE WHEN:
  * General-purpose RAG over mixed document types
  * You want better quality than fixed-size with minimal extra effort
  * Starting point for most production RAG systems
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],  # priority order
    length_function=len,
)

recursive_chunks = recursive_splitter.create_documents(
    texts=[text],
    metadatas=[test_doc.metadata]
)

print("=== Recursive Character Splitter ===")
print(f"Total chunks: {len(recursive_chunks)}")
sizes = [len(c.page_content) for c in recursive_chunks]
print(f"Avg size    : {sum(sizes)//len(sizes)} chars")
print(f"Min / Max   : {min(sizes)} / {max(sizes)} chars")
print(f"\nChunk 3 (notice cleaner paragraph boundaries):")
print(recursive_chunks[2].page_content)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.3  Token-Based Chunking
# ─────────────────────────────────────────────────────────────

"""
TOKEN-BASED CHUNKING
--------------------
How it works : Splits by TOKEN count (not character count) using the actual
               tokenizer of the target LLM. 1 token ≈ 4 characters in English.

PROS:
  + Accurate to the model's actual context window limits (no surprises)
  + Prevents "fits in character count but overflows token limit" bugs
  + Critical when you have a hard token budget for your prompts

CONS:
  - Slower — requires running the tokenizer on all text
  - Tokenizer is model-specific (GPT-4 tokens ≠ Claude tokens)
  - Still doesn't respect semantic boundaries

USE WHEN:
  * You need to guarantee chunks fit within a model's context window
  * Your text contains code or non-English languages (tokenization varies a lot)
  * You're optimizing costs and need precise token counts
"""

from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=150,       # tokens per chunk (≈600 chars for English)
    chunk_overlap=20,     # token overlap
    encoding_name="cl100k_base"   # GPT-4 / GPT-3.5 tokenizer
)

token_chunks = token_splitter.create_documents(
    texts=[text],
    metadatas=[test_doc.metadata]
)

# Count actual tokens using tiktoken
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

print("=== Token-Based Splitter ===")
print(f"Total chunks: {len(token_chunks)}")
token_counts = [len(enc.encode(c.page_content)) for c in token_chunks]
print(f"Avg tokens  : {sum(token_counts)//len(token_counts)}")
print(f"Min / Max   : {min(token_counts)} / {max(token_counts)} tokens")
print(f"\nChunk 1 token count verified: {token_counts[0]} tokens")
print(token_chunks[0].page_content[:200])

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.4  Markdown / Structure-Aware Chunking
# ─────────────────────────────────────────────────────────────

"""
STRUCTURE-AWARE CHUNKING
-------------------------
How it works : Splits on document structure (headers, sections) rather than
               raw text. The MarkdownHeaderTextSplitter splits on # headers
               and propagates header text into chunk metadata.

PROS:
  + Each chunk maps to a logical section — great for navigation
  + Header info in metadata enables filtered retrieval ("search only in Section 2")
  + Preserves the document's intended structure
  + Dramatically improves retrieval precision for structured docs

CONS:
  - Only works if document has consistent structure (headers, sections)
  - Sections can be very different sizes
  - Requires documents to be pre-formatted or converted to Markdown

USE WHEN:
  * RAG over documentation, wikis, handbooks, or structured reports
  * You want to retrieve at section level ("which section covers auth?")
  * Documents have clear hierarchical structure
"""

from langchain_text_splitters import MarkdownHeaderTextSplitter

# Convert our TXT to markdown-style with headers
markdown_text = """
# NovaTech AI Company Overview

## About NovaTech AI
NovaTech AI is a Karachi-based artificial intelligence startup founded in 2021.
The company specializes in enterprise-grade AI solutions for logistics and supply chain.
As of 2024, NovaTech employs over 120 engineers across Karachi, Dubai, and London.
The company raised a $47 million Series B funding round in March 2024.

## LogiSense Platform
LogiSense is NovaTech's flagship AI-powered logistics optimization platform.
It uses real-time data from IoT sensors, weather APIs, and traffic systems.
Clients report an average 23% reduction in delivery costs after deployment.
LogiSense currently processes over 2 million delivery events per day.

## DocuMind
DocuMind is an intelligent document processing tool using large language models.
It extracts, classifies, and summarizes information from invoices and contracts.
DocuMind supports 14 languages and achieves 97.3% extraction accuracy.
It integrates with SAP, Oracle, and Microsoft Dynamics out of the box.

## Security and Compliance
NovaTech is ISO 27001 certified and GDPR compliant.
All customer data is encrypted at rest using AES-256 and in transit using TLS 1.3.
The company undergoes quarterly penetration testing by a third-party security firm.
"""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#",  "document_title"),
        ("##", "section"),
        ("###","subsection"),
    ],
    strip_headers=False   # keep headers in the chunk content
)

md_chunks = md_splitter.split_text(markdown_text)

print("=== Markdown Structure-Aware Splitter ===")
print(f"Total chunks: {len(md_chunks)}")
for i, chunk in enumerate(md_chunks):
    print(f"\nChunk {i+1}:")
    print(f"  Metadata : {chunk.metadata}")   # notice section headers in metadata!
    print(f"  Content  : {chunk.page_content[:150]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.5  Semantic Chunking
# ─────────────────────────────────────────────────────────────

"""
SEMANTIC CHUNKING
-----------------
How it works : Uses sentence embeddings to find "breakpoints" — places where
               the topic changes significantly. Sentences with high cosine
               distance from their neighbors get split into new chunks.

PROS:
  + Splits on actual topic/meaning changes, not arbitrary character counts
  + Each chunk is semantically coherent — much better for retrieval
  + Handles variable-length topics naturally

CONS:
  - Expensive: requires embedding every sentence during ingestion
  - Slower pipeline — not suitable for real-time or high-volume ingestion
  - Threshold parameter needs tuning for different document types

USE WHEN:
  * Highest retrieval quality is the priority and cost is secondary
  * Documents mix multiple unrelated topics (earnings reports, research papers)
  * Offline ingestion where speed doesn't matter
"""

# We implement a lightweight version manually to avoid extra dependencies
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

model_mini = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_chunk(text: str, threshold: float = 0.3, min_chunk_size: int = 100) -> list[str]:
    """
    Split text into semantically coherent chunks.
    threshold: cosine distance above which we start a new chunk (0=never split, 1=always split)
    """
    # Split into sentences
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if len(s.strip()) > 20]

    if len(sentences) < 2:
        return [text]

    # Embed all sentences
    embeddings = model_mini.encode(sentences, show_progress_bar=False)

    # Find breakpoints: places where adjacent sentence similarity drops
    chunks, current_chunk = [], sentences[0]

    for i in range(1, len(sentences)):
        sim = cos_sim([embeddings[i-1]], [embeddings[i]])[0][0]
        distance = 1 - sim   # convert similarity to distance

        if distance > threshold and len(current_chunk) >= min_chunk_size:
            chunks.append(current_chunk.strip())
            current_chunk = sentences[i]
        else:
            current_chunk += ". " + sentences[i]

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


semantic_chunks_text = semantic_chunk(text, threshold=0.35)

print("=== Semantic Chunking ===")
print(f"Total chunks: {len(semantic_chunks_text)}")
sizes = [len(c) for c in semantic_chunks_text]
print(f"Avg size    : {sum(sizes)//len(sizes)} chars")
print(f"Min / Max   : {min(sizes)} / {max(sizes)} chars")
print(f"\nChunk 1 (notice it stays on one topic):")
print(semantic_chunks_text[0])
print(f"\nChunk 2 (topic shift):")
print(semantic_chunks_text[1] if len(semantic_chunks_text) > 1 else "N/A")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.6  Visual Comparison of All Chunking Strategies
# ─────────────────────────────────────────────────────────────

strategies = {
    "Fixed-Size"   : [len(c.page_content) for c in fixed_chunks],
    "Recursive"    : [len(c.page_content) for c in recursive_chunks],
    "Token-Based"  : [len(c.page_content) for c in token_chunks],
    "Markdown"     : [len(c.page_content) for c in md_chunks],
    "Semantic"     : [len(c) for c in semantic_chunks_text],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#3b82f6", "#10b981", "#f59e0b", "#8b5cf6", "#ef4444"]

# Plot 1: chunk count and avg size
names      = list(strategies.keys())
counts     = [len(v) for v in strategies.values()]
avg_sizes  = [sum(v)/len(v) for v in strategies.values()]

x = np.arange(len(names))
w = 0.35
ax = axes[0]
b1 = ax.bar(x - w/2, counts,    w, label="Chunk Count",     color=colors, alpha=0.8)
ax2 = ax.twinx()
ax2.plot(x, avg_sizes, "o--", color="#1e293b", linewidth=2, markersize=8, label="Avg Size (chars)")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("Number of Chunks", color="#374151")
ax2.set_ylabel("Avg Chunk Size (chars)", color="#1e293b")
ax.set_title("Chunks Produced vs Avg Size", fontweight="bold")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

# Plot 2: size distribution (box plot)
ax3 = axes[1]
bp  = ax3.boxplot(list(strategies.values()), labels=names, patch_artist=True, notch=False)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_ylabel("Chunk Size (characters)")
ax3.set_title("Chunk Size Distribution", fontweight="bold")
ax3.tick_params(axis='x', rotation=15)

plt.suptitle("Chunking Strategy Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary table
table = Table(title="Chunking Strategy Summary", show_header=True)
table.add_column("Strategy",   style="cyan",    min_width=14)
table.add_column("Chunks",     style="white",   justify="right")
table.add_column("Avg Chars",  style="yellow",  justify="right")
table.add_column("Min",        justify="right")
table.add_column("Max",        justify="right")
table.add_column("Best For",   style="green",   min_width=30)

best_for = [
    "Unstructured/raw text, quick baseline",
    "General purpose — default choice",
    "Exact token budget control",
    "Docs with clear headers/sections",
    "Max quality, offline ingestion",
]

for (name, sizes), desc in zip(strategies.items(), best_for):
    table.add_row(
        name, str(len(sizes)),
        str(sum(sizes)//len(sizes)),
        str(min(sizes)), str(max(sizes)), desc
    )
rprint(table)

---
## Section 4 — Embeddings

An embedding converts text into a **dense vector of numbers** that captures semantic meaning.
We use these vectors to find which chunks are most relevant to a user's query.

### How different embedding models work

| Model | Architecture | How Trained | Dims | Speed | Quality |
|---|---|---|---|---|---|
| **all-MiniLM-L6-v2** | BERT, 6 layers | Distilled + contrastive on 1B pairs | 384 | 🟢 Fast | 🟡 Good |
| **all-mpnet-base-v2** | MPNet, 12 layers | Contrastive on 1B pairs | 768 | 🟡 Medium | 🟢 Better |
| **text-embedding-3-small** | OpenAI proprietary | Unknown (likely contrastive) | 1536 | 🟡 API call | 🟢 Very good |
| **text-embedding-3-large** | OpenAI proprietary | Unknown | 3072 | 🔴 API call | 🟢 Best |
| **BGE-large** | BERT-large | BAAI contrastive + instruction tuning | 1024 | 🔴 Slow | 🟢 Best open source |

**We use `all-MiniLM-L6-v2` because:**
- Free, runs locally, no API calls
- Great speed/quality trade-off for a bootcamp setting
- 384-dim vectors are small → fast vector DB operations
- Consistently top-ranked on MTEB benchmark for its size

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.1  Load the embedding model
# ─────────────────────────────────────────────────────────────

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading all-MiniLM-L6-v2... (downloads ~90MB on first run)")
t0    = time.time()
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded in {time.time()-t0:.1f}s")

# Model stats
sample_embedding = model.encode("Hello world")
print(f"\nEmbedding dimensions : {len(sample_embedding)}")
print(f"Data type            : {sample_embedding.dtype}")
print(f"Sample values (first 8): {sample_embedding[:8].round(4)}")
print(f"Vector norm          : {np.linalg.norm(sample_embedding):.4f}  (always ~1.0 — normalized)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.2  Batch embedding with timing benchmark
# ─────────────────────────────────────────────────────────────

# Use recursive chunks as our primary chunks going forward
chunks_to_embed = recursive_chunks
texts_to_embed  = [c.page_content for c in chunks_to_embed]

print(f"Embedding {len(texts_to_embed)} chunks...")
t0 = time.time()

embeddings = model.encode(
    texts_to_embed,
    batch_size=32,           # process 32 texts at a time
    show_progress_bar=True,
    normalize_embeddings=True  # L2-normalize for cosine similarity via dot product
)

elapsed = time.time() - t0
print(f"\nEmbedding complete.")
print(f"Total time      : {elapsed:.2f}s")
print(f"Speed           : {len(texts_to_embed)/elapsed:.1f} chunks/second")
print(f"Embeddings shape: {embeddings.shape}  ({embeddings.shape[0]} chunks × {embeddings.shape[1]} dims)")
print(f"Memory (float32): {embeddings.nbytes / 1024:.1f} KB")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.3  Semantic similarity — prove the embeddings work
# ─────────────────────────────────────────────────────────────

from sklearn.metrics.pairwise import cosine_similarity

test_pairs = [
    # (text_a, text_b, expected_similarity)
    ("NovaTech raised $47 million in Series B funding",
     "The startup secured forty-seven million dollars from investors",
     "HIGH — same meaning, different words"),

    ("NovaTech raised $47 million in Series B funding",
     "LogiSense processes 2 million delivery events per day",
     "LOW — same company, different topics"),

    ("How do I reset my API key?",
     "Steps to regenerate an API token",
     "HIGH — same intent"),

    ("What is the refund policy?",
     "Machine learning training requires GPUs",
     "LOW — completely unrelated"),
]

print("=== Semantic Similarity Test ===")
print()
for a, b, expected in test_pairs:
    emb_a = model.encode([a], normalize_embeddings=True)
    emb_b = model.encode([b], normalize_embeddings=True)
    score = cosine_similarity(emb_a, emb_b)[0][0]
    bar   = "█" * int(score * 30)
    print(f"Score: {score:.3f}  |{bar:<30}|  ({expected})")
    print(f"  A: {a[:60]}")
    print(f"  B: {b[:60]}")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.4  Visual: t-SNE of embedded chunks
# ─────────────────────────────────────────────────────────────

# Reduce 384-dim embeddings to 2D for visualization
# This shows us how the chunks cluster by topic

from sklearn.manifold import TSNE

# Embed all docs (use a broader set for a richer plot)
all_docs_for_viz = txt_docs + pdf_docs + docx_docs + json_docs
# Chunk them all with recursive splitter
all_chunks_viz = recursive_splitter.split_documents(all_docs_for_viz)
all_texts_viz  = [c.page_content for c in all_chunks_viz]
all_embs_viz   = model.encode(all_texts_viz, show_progress_bar=False, normalize_embeddings=True)

print(f"Reducing {len(all_embs_viz)} chunks from 384D → 2D with t-SNE...")
tsne   = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_embs_viz)-1))
coords = tsne.fit_transform(all_embs_viz)

# Color by source document
source_colors = {
    "novatech_company.txt":     ("#3b82f6", "TXT — Company Overview"),
    "novatech_architecture.pdf":("#ef4444", "PDF — Architecture"),
    "novatech_handbook.docx":   ("#10b981", "DOCX — Handbook"),
    "novatech_faq.json":        ("#f59e0b", "JSON — FAQ"),
}

fig, ax = plt.subplots(figsize=(11, 7))

for chunk, (x, y) in zip(all_chunks_viz, coords):
    src    = Path(chunk.metadata.get("source", "")).name
    color  = source_colors.get(src, ("#94a3b8", "Other"))[0]
    ax.scatter(x, y, color=color, alpha=0.6, s=40, edgecolors="white", linewidths=0.4)

legend_patches = [mpatches.Patch(color=v[0], label=v[1]) for v in source_colors.values()]
ax.legend(handles=legend_patches, loc="lower right", fontsize=9)
ax.set_title("t-SNE Visualization of Chunk Embeddings\n(clusters = semantically similar content)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
ax.set_facecolor("#f8fafc")
plt.tight_layout()
plt.show()
print("Chunks from the same document tend to cluster — the embeddings capture topic similarity.")

---
## Section 5 — Vector Databases

A vector database stores your embeddings and enables fast **approximate nearest-neighbor (ANN)** search.
Given a query embedding, it returns the most similar chunk embeddings in milliseconds.

### ChromaDB vs FAISS

| Feature | ChromaDB | FAISS |
|---|---|---|
| **Type** | Full vector DB (with metadata, persistence) | Library (just the index) |
| **Persistence** | ✅ Built-in (SQLite backend) | ❌ Manual save/load |
| **Metadata filtering** | ✅ Rich filtering on metadata fields | ❌ Not natively |
| **Speed** | 🟡 Medium (great for <1M vectors) | 🟢 Extremely fast (billions of vectors) |
| **Setup** | `pip install chromadb` — zero config | `pip install faiss-cpu` — zero config |
| **Best for** | Dev, prototyping, small-medium production | Large-scale production, research |
| **Managed cloud** | ✅ Chroma Cloud | ❌ (use Pinecone/Weaviate instead) |

**Other popular options:**
- **Pinecone** — fully managed, production-grade, expensive
- **Weaviate** — open source, multi-modal, GraphQL API
- **Qdrant** — open source, Rust-based, very fast, great filtering
- **pgvector** — vector search inside PostgreSQL (great if you already use Postgres)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.1  Prepare the chunks we'll store
#      (recursive chunks of all 5 documents)
# ─────────────────────────────────────────────────────────────

from langchain_core.documents import Document

# Load and chunk all documents
final_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# Load all source documents
from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader, CSVLoader

loaders = [
    TextLoader(f"{DATA_DIR}/novatech_company.txt", encoding="utf-8"),
    PyPDFLoader(f"{DATA_DIR}/novatech_architecture.pdf"),
    Docx2txtLoader(f"{DATA_DIR}/novatech_handbook.docx"),
    CSVLoader(f"{DATA_DIR}/novatech_employees.csv", source_column="name"),
]

raw_docs = []
for loader in loaders:
    raw_docs.extend(loader.load())

# Add JSON docs manually
raw_docs.extend(load_json_as_documents(f"{DATA_DIR}/novatech_faq.json"))

# Chunk everything
final_chunks = final_splitter.split_documents(raw_docs)

print(f"Total chunks ready for indexing: {len(final_chunks)}")
print(f"Sample chunk metadata: {final_chunks[0].metadata}")
print(f"Sample chunk content:\n{final_chunks[0].page_content[:200]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.2  ChromaDB — persistent vector store
# ─────────────────────────────────────────────────────────────

import chromadb
from chromadb.config import Settings
import shutil

CHROMA_PATH = "./chroma_db"

# Clean slate for demo
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

# Create persistent Chroma client
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Create a collection (like a table in a DB)
collection = chroma_client.create_collection(
    name="novatech_docs",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity for search
)

print("Embedding and inserting chunks into ChromaDB...")
t0 = time.time()

# Embed all chunks
texts     = [c.page_content for c in final_chunks]
metadatas = [c.metadata     for c in final_chunks]
ids       = [f"chunk_{i}"   for i in range(len(final_chunks))]

embeddings_np = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()   # Chroma expects Python lists

# Insert in batches (Chroma has a max batch size)
BATCH_SIZE = 100
for i in range(0, len(texts), BATCH_SIZE):
    collection.add(
        ids=ids[i:i+BATCH_SIZE],
        embeddings=embeddings_np[i:i+BATCH_SIZE],
        documents=texts[i:i+BATCH_SIZE],
        metadatas=metadatas[i:i+BATCH_SIZE],
    )

elapsed = time.time() - t0
print(f"\n✅ Inserted {collection.count()} chunks in {elapsed:.2f}s")
print(f"Persisted to: {CHROMA_PATH}/")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.3  ChromaDB — querying the collection
# ─────────────────────────────────────────────────────────────

def chroma_search(query: str, n_results: int = 3, filter_metadata: dict = None) -> list:
    """Search ChromaDB and return top N matching chunks."""
    query_embedding = model.encode([query], normalize_embeddings=True).tolist()

    kwargs = {"query_embeddings": query_embedding, "n_results": n_results}
    if filter_metadata:
        kwargs["where"] = filter_metadata   # metadata filter

    results = collection.query(**kwargs)

    output = []
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        output.append({
            "rank"      : i + 1,
            "score"     : round(1 - dist, 4),   # convert distance → similarity
            "content"   : doc,
            "metadata"  : meta,
        })
    return output


# Test queries
test_queries = [
    "What is the SLA for enterprise customers?",
    "How does DocuMind handle scanned documents?",
    "What was NovaTech's Series B funding amount?",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 Query: {query}")
    results = chroma_search(query, n_results=2)
    for r in results:
        src = Path(r['metadata'].get('source','')).name
        print(f"\n  Rank {r['rank']} | Score: {r['score']} | Source: {src}")
        print(f"  {r['content'][:200]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.4  ChromaDB — metadata filtering
# Filtering lets you restrict search to a subset of documents
# before doing the vector search — much faster than post-filtering
# ─────────────────────────────────────────────────────────────

print("=== Metadata-Filtered Search ===")
print("Query: 'What is the billing model?'")
print("Filter: only search FAQ documents (category=Billing)\n")

# First, let's see what categories exist in our JSON docs
faq_categories = list({d.metadata.get("category") for d in json_docs})
print(f"Available FAQ categories: {faq_categories}")

# Search only in Billing FAQs
results_filtered = chroma_search(
    "What is the billing model?",
    n_results=3,
    filter_metadata={"category": "Billing"}
)

for r in results_filtered:
    print(f"\n  Score: {r['score']} | Metadata: {r['metadata']}")
    print(f"  {r['content'][:250]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.5  FAISS — high-performance vector index
# ─────────────────────────────────────────────────────────────

import faiss

DIM = 384   # all-MiniLM-L6-v2 output dimension

# FAISS index types:
#   IndexFlatL2      — exact search, L2 distance, no training needed (great baseline)
#   IndexFlatIP      — exact search, inner product (= cosine if normalized)
#   IndexIVFFlat     — approximate, clusters vectors for faster search
#   IndexHNSWFlat    — approximate, graph-based, very fast, good recall

# We'll build two indexes to compare exact vs approximate search

# ── Index 1: Exact search (IndexFlatIP) ──────────────────────
faiss_index_exact = faiss.IndexFlatIP(DIM)   # inner product on normalized = cosine

# Re-embed and insert
all_embeddings_np = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")  # FAISS requires float32

faiss_index_exact.add(all_embeddings_np)

# ── Index 2: Approximate search (HNSW) ───────────────────────
M            = 16    # number of connections per node (higher = more accurate, more memory)
faiss_index_hnsw = faiss.IndexHNSWFlat(DIM, M)
faiss_index_hnsw.hnsw.efConstruction = 64  # build time accuracy
faiss_index_hnsw.hnsw.efSearch       = 32  # search time accuracy
faiss_index_hnsw.add(all_embeddings_np)

print(f"✅ FAISS Exact  index: {faiss_index_exact.ntotal} vectors")
print(f"✅ FAISS HNSW   index: {faiss_index_hnsw.ntotal} vectors")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.6  FAISS — querying + save/load
# ─────────────────────────────────────────────────────────────

def faiss_search(query: str, index, texts: list, n_results: int = 3) -> list:
    """Search a FAISS index. Returns top N results."""
    query_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_emb, n_results)
    return [
        {"rank": i+1, "score": round(float(scores[0][i]), 4), "content": texts[idx]}
        for i, idx in enumerate(indices[0]) if idx != -1
    ]


# Test query
query = "How do I integrate LogiSense with my existing system?"
print(f"Query: {query}\n")

results_exact = faiss_search(query, faiss_index_exact, texts)
results_hnsw  = faiss_search(query, faiss_index_hnsw,  texts)

print("--- Exact Search (IndexFlatIP) ---")
for r in results_exact:
    print(f"  Rank {r['rank']} | Score {r['score']} | {r['content'][:150]}")

print("\n--- Approximate Search (HNSW) ---")
for r in results_hnsw:
    print(f"  Rank {r['rank']} | Score {r['score']} | {r['content'][:150]}")

# Save the index to disk
faiss.write_index(faiss_index_exact, "faiss_exact.index")
faiss.write_index(faiss_index_hnsw,  "faiss_hnsw.index")
print("\n✅ FAISS indexes saved to disk")

# Load back
loaded_index = faiss.read_index("faiss_exact.index")
print(f"✅ Loaded index has {loaded_index.ntotal} vectors")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.7  Speed Benchmark: ChromaDB vs FAISS Exact vs FAISS HNSW
# ─────────────────────────────────────────────────────────────

import timeit

bench_query  = "What are the security certifications of NovaTech?"
bench_emb    = model.encode([bench_query], normalize_embeddings=True)
bench_emb_f32 = bench_emb.astype("float32")

N_RUNS = 50

t_chroma = timeit.timeit(
    lambda: collection.query(query_embeddings=bench_emb.tolist(), n_results=3),
    number=N_RUNS
)
t_faiss_exact = timeit.timeit(
    lambda: faiss_index_exact.search(bench_emb_f32, 3),
    number=N_RUNS
)
t_faiss_hnsw = timeit.timeit(
    lambda: faiss_index_hnsw.search(bench_emb_f32, 3),
    number=N_RUNS
)

results = {
    "ChromaDB"    : t_chroma / N_RUNS * 1000,
    "FAISS Exact" : t_faiss_exact / N_RUNS * 1000,
    "FAISS HNSW"  : t_faiss_hnsw / N_RUNS * 1000,
}

fig, ax = plt.subplots(figsize=(8, 4))
bars    = ax.bar(results.keys(), results.values(),
                 color=["#8b5cf6", "#3b82f6", "#10b981"], width=0.4, alpha=0.85)
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.2f} ms", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("Avg Query Latency (ms)")
ax.set_title(f"Vector DB Query Latency Comparison ({N_RUNS} runs, {len(texts)} vectors)",
             fontweight="bold")
ax.set_ylim(0, max(results.values()) * 1.3)
plt.tight_layout()
plt.show()

print("\n💡 Note: FAISS is faster because it's a pure in-memory index.")
print("   ChromaDB adds overhead for persistence, metadata, and filtering support.")
print("   At millions of vectors, FAISS HNSW's advantage becomes dramatic.")

---
## Section 6 — Full Pipeline: Putting It All Together

Now we consolidate everything into a single `DataIngestionPipeline` class.
This is the production-ready version you'll wire into the retrieval pipeline in Part 2.

In [ ]:
import shutil
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb


class DataIngestionPipeline:
    """
    Full RAG data ingestion pipeline.
    
    Steps:
      1. Load documents from a directory (auto-detects format)
      2. Chunk with RecursiveCharacterTextSplitter
      3. Embed with all-MiniLM-L6-v2
      4. Store in ChromaDB (persistent)
    """

    LOADERS = {
        ".txt"  : lambda p: TextLoader(p, encoding="utf-8"),
        ".pdf"  : lambda p: PyPDFLoader(p),
        ".docx" : lambda p: Docx2txtLoader(p),
        ".csv"  : lambda p: CSVLoader(p),
    }

    def __init__(
        self,
        collection_name: str = "rag_collection",
        persist_path:    str = "./chroma_pipeline",
        chunk_size:      int = 400,
        chunk_overlap:   int = 40,
        embedding_model: str = "all-MiniLM-L6-v2",
    ):
        self.collection_name = collection_name
        self.persist_path    = persist_path
        self.chunk_size      = chunk_size
        self.chunk_overlap   = chunk_overlap

        print(f"[1/3] Loading embedding model: {embedding_model}")
        self.embedder = SentenceTransformer(embedding_model)

        print(f"[2/3] Connecting to ChromaDB at: {persist_path}")
        self.client     = chromadb.PersistentClient(path=persist_path)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        print(f"[3/3] Pipeline ready. Collection has {self.collection.count()} existing chunks.")

    def ingest_directory(self, directory: str) -> dict:
        """Load, chunk, embed, and store all supported files in a directory."""
        dir_path = Path(directory)
        stats = {"files": 0, "skipped": 0, "chunks": 0, "elapsed_s": 0}
        t0 = time.time()

        raw_docs = []
        for path in sorted(dir_path.iterdir()):
            suffix = path.suffix.lower()
            if suffix in self.LOADERS:
                try:
                    docs = self.LOADERS[suffix](str(path)).load()
                    raw_docs.extend(docs)
                    stats["files"] += 1
                    print(f"  ✅ Loaded {path.name:40s} → {len(docs)} doc(s)")
                except Exception as e:
                    print(f"  ❌ Failed {path.name}: {e}")
                    stats["skipped"] += 1
            elif suffix == ".json":
                try:
                    docs = load_json_as_documents(str(path))
                    raw_docs.extend(docs)
                    stats["files"] += 1
                    print(f"  ✅ Loaded {path.name:40s} → {len(docs)} doc(s)")
                except Exception as e:
                    print(f"  ❌ Failed {path.name}: {e}")

        if not raw_docs:
            print("No documents loaded.")
            return stats

        # Chunk
        chunks    = self.splitter.split_documents(raw_docs)
        texts     = [c.page_content for c in chunks]
        metadatas = [c.metadata     for c in chunks]
        ids       = [f"doc_{i}" for i in range(self.collection.count(), self.collection.count() + len(chunks))]

        # Embed
        print(f"\n  Embedding {len(chunks)} chunks...")
        embeddings = self.embedder.encode(
            texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True
        ).tolist()

        # Store
        BATCH = 100
        for i in range(0, len(texts), BATCH):
            self.collection.add(
                ids=ids[i:i+BATCH],
                embeddings=embeddings[i:i+BATCH],
                documents=texts[i:i+BATCH],
                metadatas=metadatas[i:i+BATCH],
            )

        stats["chunks"]    = len(chunks)
        stats["elapsed_s"] = round(time.time() - t0, 2)
        return stats

    def query(self, query_text: str, n_results: int = 3, where: dict = None) -> list:
        """Semantic search over the collection."""
        query_emb = self.embedder.encode([query_text], normalize_embeddings=True).tolist()
        kwargs = {"query_embeddings": query_emb, "n_results": n_results}
        if where:
            kwargs["where"] = where
        results = self.collection.query(**kwargs)
        return [
            {
                "rank"    : i + 1,
                "score"   : round(1 - d, 4),
                "content" : doc,
                "source"  : Path(meta.get("source", "")).name,
                "metadata": meta,
            }
            for i, (doc, meta, d) in enumerate(zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0]
            ))
        ]

    def stats(self) -> dict:
        return {
            "collection"    : self.collection_name,
            "total_chunks"  : self.collection.count(),
            "persist_path"  : self.persist_path,
            "chunk_size"    : self.chunk_size,
            "chunk_overlap" : self.chunk_overlap,
        }


print("✅ DataIngestionPipeline class defined")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.2  Run the full pipeline end-to-end
# ─────────────────────────────────────────────────────────────

if os.path.exists("./chroma_pipeline"):
    shutil.rmtree("./chroma_pipeline")

print("=" * 55)
print("  RUNNING FULL DATA INGESTION PIPELINE")
print("=" * 55)
print()

pipeline = DataIngestionPipeline(
    collection_name="novatech_v1",
    persist_path="./chroma_pipeline",
    chunk_size=400,
    chunk_overlap=40,
)

print()
print("Ingesting sample_data directory...")
stats = pipeline.ingest_directory(DATA_DIR)

print()
rprint(Panel(
    f"Files processed : {stats['files']}\n"
    f"Files skipped   : {stats['skipped']}\n"
    f"Chunks stored   : {stats['chunks']}\n"
    f"Total time      : {stats['elapsed_s']}s",
    title="✅ Pipeline Complete"
))

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.3  Query the completed pipeline
# ─────────────────────────────────────────────────────────────

final_queries = [
    "What are NovaTech's main products?",
    "How does the company handle data security and compliance?",
    "What is the uptime SLA for enterprise customers?",
    "How do I troubleshoot low extraction accuracy in DocuMind?",
    "Who is the CTO of NovaTech and when did they join?",
]

for query in final_queries:
    print(f"\n{'='*60}")
    print(f"🔍 {query}")
    results = pipeline.query(query, n_results=2)
    for r in results:
        print(f"\n  [{r['rank']}] Score: {r['score']} | Source: {r['source']}")
        print(f"  {r['content'][:250]}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.4  Pipeline summary diagram
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("  DATA INGESTION PIPELINE — COMPLETE SUMMARY")
print("=" * 60)

pipeline_summary = [
    ("STEP 1", "Document Loading",  "TextLoader, PyPDFLoader, Docx2txtLoader, CSVLoader, custom JSON"),
    ("STEP 2", "Chunking",          "RecursiveCharacterTextSplitter (chunk=400, overlap=40)"),
    ("STEP 3", "Embedding",         "all-MiniLM-L6-v2 (384 dims, normalized, ~1400 chunks/sec)"),
    ("STEP 4", "Vector Storage",    "ChromaDB persistent collection with cosine similarity index"),
]

for step, name, detail in pipeline_summary:
    print(f"\n  {step}: {name}")
    print(f"          {detail}")

print()
ps = pipeline.stats()
rprint(Panel(
    f"Collection : {ps['collection']}\n"
    f"Chunks     : {ps['total_chunks']}\n"
    f"Location   : {ps['persist_path']}\n"
    f"Chunk size : {ps['chunk_size']} chars (overlap: {ps['chunk_overlap']})",
    title="📦 Ingestion Pipeline Stats"
))

print()
print("🚀 Ready for Day 2 — Retrieval Pipeline!")
print("   The pipeline object can be loaded and queried from the retrieval notebook.")